# 00 — Conversão de Dados: EDF → Espectogramas STFT

**Projeto:** epilepsy-detection  
**Dataset:** Siena Scalp EEG Database (PhysioNet)  
**Objetivo:** Converter sinais EEG de 14 pacientes epilépticos em espectogramas STFT
classificados em 4 classes: `pre_ictal`, `ictal`, `pos_ictal`, `normal`.

---

## Fluxo do Notebook
```
0. Setup & Constantes
1. Inventário de Dados
2. Parse das Anotações
3. Extração de Segmentos
4. Geração dos Espectogramas
5. Geração do annotations.csv
6. Relatório
7. Visualização
```


## 0. Setup & Constantes


In [1]:
import sys, os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# ── Adiciona raiz do projeto ao sys.path ──────────────────────────────────
PROJECT_ROOT = Path(globals()['_dh'][0]).parent  # pasta do notebook -> raiz
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    load_edf, STANDARD_CHANNELS,
    parse_seizure_list,
    extract_segments,
    segment_to_spectrogram,
    save_spectrogram,
)
from src.preprocessing.stft_converter import plot_class_grid

print(f'Raiz do projeto: {PROJECT_ROOT}')


Raiz do projeto: /home/orlandomota/Documents/repositorios_github/epilepsy-detection


In [2]:
# ── Constantes do Pipeline ────────────────────────────────────────────────
DATA_ROOT    = PROJECT_ROOT / 'physionet.org' / 'files' / 'siena-scalp-eeg' / '1.0.0'
OUTPUT_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'spectrograms'
ANNOT_DIR    = PROJECT_ROOT / 'data' / 'annotations'
WINDOW_S     = 90    # janela fixa em segundos
FS           = 512   # taxa de amostragem (Hz)
NPERSEG      = 256   # tamanho da janela STFT
NOVERLAP     = 128   # sobreposicao STFT

# Pacientes utilizados (PN10 excluido por ter 20 canais)
PATIENTS = ['PN00', 'PN01', 'PN03', 'PN05', 'PN06', 'PN07',
            'PN09', 'PN11', 'PN12', 'PN13', 'PN14', 'PN16', 'PN17']

CLASSES  = ['pre_ictal', 'ictal', 'pos_ictal', 'normal']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT  : {DATA_ROOT}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'Pacientes  : {len(PATIENTS)}')
print(f'Janela     : {WINDOW_S}s = {WINDOW_S * FS:,} amostras')


DATA_ROOT  : /home/orlandomota/Documents/repositorios_github/epilepsy-detection/physionet.org/files/siena-scalp-eeg/1.0.0
OUTPUT_DIR : /home/orlandomota/Documents/repositorios_github/epilepsy-detection/data/processed/spectrograms
Pacientes  : 13
Janela     : 90s = 46,080 amostras


## 1. Inventário de Dados
Verifica quais pacientes possuem arquivos `.edf` e `Seizures-list-PNxx.txt` disponíveis localmente.


In [3]:
inventory = []

for patient in PATIENTS:
    patient_dir  = DATA_ROOT / patient
    edf_files    = sorted(patient_dir.glob('*.edf')) if patient_dir.exists() else []
    seizure_txt  = patient_dir / f'Seizures-list-{patient}.txt'

    inventory.append({
        'patient_id'      : patient,
        'dir_exists'      : patient_dir.exists(),
        'n_edf_files'     : len(edf_files),
        'edf_files'       : [f.name for f in edf_files],
        'has_seizure_list': seizure_txt.exists(),
        'seizure_list_path': str(seizure_txt),
    })

df_inv = pd.DataFrame(inventory)
display(df_inv[['patient_id', 'dir_exists', 'n_edf_files', 'has_seizure_list']])

ready     = df_inv[(df_inv.n_edf_files > 0) & (df_inv.has_seizure_list)]
no_edf    = df_inv[df_inv.n_edf_files == 0]
no_annot  = df_inv[(df_inv.n_edf_files > 0) & (~df_inv.has_seizure_list)]

print(f'\n✅ Prontos para processar : {len(ready)} pacientes -> {list(ready.patient_id)}')
print(f'⚠️  Sem EDF               : {len(no_edf)} pacientes -> {list(no_edf.patient_id)}')
print(f'⚠️  Sem anotacoes         : {len(no_annot)} pacientes -> {list(no_annot.patient_id)}')


,patient_id,dir_exists,n_edf_files,has_seizure_list
0,PN00,True,5,True
1,PN01,True,1,False
2,PN03,True,0,False
3,PN05,True,0,False
4,PN06,True,0,False
5,PN07,True,0,False
6,PN09,True,0,False
7,PN11,True,0,False
8,PN12,True,0,False
9,PN13,True,0,False



✅ Prontos para processar : 1 pacientes -> ['PN00']
⚠️  Sem EDF               : 11 pacientes -> ['PN03', 'PN05', 'PN06', 'PN07', 'PN09', 'PN11', 'PN12', 'PN13', 'PN14', 'PN16', 'PN17']
⚠️  Sem anotacoes         : 1 pacientes -> ['PN01']


## 2. Parse das Anotações
Lê os arquivos `Seizures-list-PNxx.txt` para cada paciente disponível.


In [4]:
all_annotations = {}  # patient_id -> list of seizure dicts

for _, row in df_inv[df_inv.has_seizure_list].iterrows():
    pid = row['patient_id']
    try:
        seizures = parse_seizure_list(row['seizure_list_path'])
        all_annotations[pid] = seizures
        print(f'[{pid}] {len(seizures)} crise(s) encontrada(s)')
        for sz in seizures:
            print(f"  Crise {sz['seizure_n']:2d} | {sz['edf_file']}"
                  f" | Inicio: {sz['sz_start_rel_s']:7.1f}s"
                  f" | Duracao: {sz['sz_duration_s']:.1f}s")
    except Exception as e:
        print(f'[{pid}] ERRO ao parsear anotacoes: {e}')
        all_annotations[pid] = []

print(f'\nTotal de pacientes com anotacoes: {len(all_annotations)}')
total_sz = sum(len(v) for v in all_annotations.values())
print(f'Total de crises mapeadas        : {total_sz}')


[PN00] 5 crise(s) encontrada(s)
  Crise  1 | PN00-1.edf | Inicio:  1143.0s | Duracao: 70.0s
  Crise  2 | PN00-2.edf | Inicio:  1220.0s | Duracao: 54.0s
  Crise  3 | PN00-3.edf | Inicio:   765.0s | Duracao: 3660.0s
  Crise  4 | PN00-4.edf | Inicio:  1006.0s | Duracao: 74.0s
  Crise  5 | PN00-5.edf | Inicio:   904.0s | Duracao: 67.0s

Total de pacientes com anotacoes: 1
Total de crises mapeadas        : 5


## 3. Extração de Segmentos & Geração de Espectogramas
Para cada paciente → cada EDF → cada crise → extrai 4 segmentos de 90 s → converte em PNG.


In [5]:
records = []  # lista de dicts para o annotations.csv

patients_to_process = df_inv[
    (df_inv.n_edf_files > 0) & (df_inv.has_seizure_list)
].to_dict('records')

for patient_info in tqdm(patients_to_process, desc='Pacientes'):
    pid          = patient_info['patient_id']
    patient_dir  = DATA_ROOT / pid
    out_pid_dir  = OUTPUT_DIR / pid
    out_pid_dir.mkdir(parents=True, exist_ok=True)

    seizures = all_annotations.get(pid, [])
    if not seizures:
        print(f'[{pid}] Sem crises — pulando.')
        continue

    # Agrupa crises por arquivo EDF
    from collections import defaultdict
    edf_to_seizures = defaultdict(list)
    for sz in seizures:
        edf_to_seizures[sz['edf_file']].append(sz)

    for edf_name, sz_list in tqdm(edf_to_seizures.items(), desc=f'  {pid} EDFs', leave=False):
        edf_path = patient_dir / edf_name
        if not edf_path.exists():
            print("Edf_doesn't exist", edf_path)
            for sz in sz_list:
                for cls in CLASSES:
                    records.append({
                        'patient_id': pid, 'edf_file': edf_name,
                        'seizure_n': sz['seizure_n'], 'class': cls,
                        'spectrogram_path': None, 'status': 'edf_missing',
                    })
            continue

        # Carrega o EDF
        try:
            raw, meta = load_edf(str(edf_path))
            raw_data  = raw.get_data()  # shape (n_channels, n_samples)
        except Exception as e:
            print(f'  [{pid}/{edf_name}] ERRO ao carregar EDF: {e}')
            continue

        # Obtem lista de starts de todas as crises do mesmo EDF
        all_starts = [s['sz_start_rel_s'] for s in sz_list]

        for sz in tqdm(sz_list, desc=f'    Crises', leave=False):
            seg_dict = extract_segments(
                raw_data       = raw_data,
                sz_start_rel_s = sz['sz_start_rel_s'],
                sz_end_rel_s   = sz['sz_end_rel_s'],
                all_sz_starts  = all_starts,
                window_s       = WINDOW_S,
                fs             = meta['fs'],
            )

            for cls, segment in seg_dict.items():
                base_name = edf_name.replace('.edf', '')
                out_name  = f"{base_name}_seizure{sz['seizure_n']:02d}_{cls}.png"
                out_path  = out_pid_dir / out_name

                if segment is None:
                    status = 'skipped_no_data'
                    sp_path = None
                else:
                    try:
                        img = segment_to_spectrogram(
                            segment, fs=meta['fs'],
                            nperseg=NPERSEG, noverlap=NOVERLAP
                        )
                        save_spectrogram(img, str(out_path))
                        status  = 'ok'
                        sp_path = str(out_path.relative_to(PROJECT_ROOT))
                    except Exception as e:
                        print(f'  ERRO espectograma [{pid}/{out_name}]: {e}')
                        status  = f'error: {e}'
                        sp_path = None

                records.append({
                    'patient_id'       : pid,
                    'edf_file'         : edf_name,
                    'seizure_n'        : sz['seizure_n'],
                    'sz_start_rel_s'   : sz['sz_start_rel_s'],
                    'sz_end_rel_s'     : sz['sz_end_rel_s'],
                    'sz_duration_s'    : sz['sz_duration_s'],
                    'class'            : cls,
                    'spectrogram_path' : sp_path,
                    'status'           : status,
                })

        del raw, raw_data  # libera memoria

print(f'\nProcessamento concluido. Total de entradas: {len(records)}')


Pacientes:   0%|          | 0/1 [00:00<?, ?it/s]

  PN00 EDFs:   0%|          | 0/5 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


    Crises:   0%|          | 0/1 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


    Crises:   0%|          | 0/1 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


    Crises:   0%|          | 0/1 [00:00<?, ?it/s]


Processamento concluido. Total de entradas: 20


## 5. Geração do `annotations.csv`
Manifesto global de todos os segmentos — base para treino/validação/teste.


In [6]:
df_annot = pd.DataFrame(records)

if not df_annot.empty:
    annot_path = ANNOT_DIR / 'annotations.csv'
    df_annot.to_csv(annot_path, index=False)
    print(f'annotations.csv salvo em: {annot_path}')
    print(f'Total de linhas: {len(df_annot)}')
    display(df_annot.head(10))
else:
    print('Nenhum registro gerado. Verifique os dados disponíveis.')


annotations.csv salvo em: /home/orlandomota/Documents/repositorios_github/epilepsy-detection/data/annotations/annotations.csv
Total de linhas: 20


,patient_id,edf_file,seizure_n,sz_start_rel_s,sz_end_rel_s,sz_duration_s,class,spectrogram_path,status
0,PN00,PN00-1.edf,1,1143.0,1213.0,70.0,pre_ictal,data/processed/spectrograms/PN00/PN00-1_seizur...,ok
1,PN00,PN00-1.edf,1,1143.0,1213.0,70.0,ictal,data/processed/spectrograms/PN00/PN00-1_seizur...,ok
2,PN00,PN00-1.edf,1,1143.0,1213.0,70.0,pos_ictal,data/processed/spectrograms/PN00/PN00-1_seizur...,ok
3,PN00,PN00-1.edf,1,1143.0,1213.0,70.0,normal,data/processed/spectrograms/PN00/PN00-1_seizur...,ok
4,PN00,PN00-2.edf,2,1220.0,1274.0,54.0,pre_ictal,data/processed/spectrograms/PN00/PN00-2_seizur...,ok
5,PN00,PN00-2.edf,2,1220.0,1274.0,54.0,ictal,data/processed/spectrograms/PN00/PN00-2_seizur...,ok
6,PN00,PN00-2.edf,2,1220.0,1274.0,54.0,pos_ictal,data/processed/spectrograms/PN00/PN00-2_seizur...,ok
7,PN00,PN00-2.edf,2,1220.0,1274.0,54.0,normal,data/processed/spectrograms/PN00/PN00-2_seizur...,ok
8,PN00,PN00-3.edf,3,765.0,4425.0,3660.0,pre_ictal,data/processed/spectrograms/PN00/PN00-3_seizur...,ok
9,PN00,PN00-3.edf,3,765.0,4425.0,3660.0,ictal,data/processed/spectrograms/PN00/PN00-3_seizur...,ok


## 6. Relatório por Paciente


In [7]:
if not df_annot.empty:
    # Contagem por status
    ok_mask  = df_annot['status'] == 'ok'
    report   = []

    for pid in PATIENTS:
        pid_df   = df_annot[df_annot.patient_id == pid]
        if pid_df.empty:
            continue
        n_sz     = pid_df.seizure_n.nunique()
        n_ok     = pid_df[pid_df.status == 'ok'].shape[0]
        n_skip   = pid_df[pid_df.status != 'ok'].shape[0]
        by_class = pid_df[pid_df.status == 'ok'].groupby('class').size().to_dict()
        report.append({
            'Paciente': pid,
            'Crises'  : n_sz,
            'OK'      : n_ok,
            'Skipped' : n_skip,
            'pre_ictal' : by_class.get('pre_ictal', 0),
            'ictal'     : by_class.get('ictal', 0),
            'pos_ictal' : by_class.get('pos_ictal', 0),
            'normal'    : by_class.get('normal', 0),
        })

    df_report = pd.DataFrame(report)
    print('=== Relatório de Processamento ===')
    display(df_report)

    # Totais
    print(f"\nTotal de espectogramas gerados : {df_annot[df_annot.status=='ok'].shape[0]}")
    print(f"Total skipped / erros          : {df_annot[df_annot.status!='ok'].shape[0]}")
    print(f"\nDistribuicao por classe:")
    display(df_annot[df_annot.status=='ok'].groupby('class').size().rename('count').to_frame())
else:
    print('Sem dados para gerar relatorio.')


=== Relatório de Processamento ===


,Paciente,Crises,OK,Skipped,pre_ictal,ictal,pos_ictal,normal
0,PN00,5,19,1,5,5,4,5



Total de espectogramas gerados : 19
Total skipped / erros          : 1

Distribuicao por classe:


,count
class,
ictal,5
normal,5
pos_ictal,4
pre_ictal,5


## 7. Visualização dos Espectogramas
Exibe os 4 espectogramas da primeira crise do primeiro paciente disponível.


In [8]:
# Encontra o primeiro paciente com espectogramas gerados
if not df_annot.empty and (df_annot.status == 'ok').any():
    first_ok   = df_annot[df_annot.status == 'ok'].iloc[0]
    vis_pid    = first_ok['patient_id']
    vis_edf    = first_ok['edf_file']
    vis_sz     = first_ok['seizure_n']

    # Carrega as 4 imagens desse conjunto
    from PIL import Image
    spects = {}
    for cls in CLASSES:
        row = df_annot[
            (df_annot.patient_id == vis_pid) &
            (df_annot.edf_file   == vis_edf) &
            (df_annot.seizure_n  == vis_sz)  &
            (df_annot['class']   == cls)     &
            (df_annot.status     == 'ok')
        ]
        if not row.empty and row.iloc[0]['spectrogram_path']:
            img_path = PROJECT_ROOT / row.iloc[0]['spectrogram_path']
            if img_path.exists():
                spects[cls] = np.array(Image.open(img_path))
            else:
                spects[cls] = None
        else:
            spects[cls] = None

    fig = plot_class_grid(spects, vis_pid, vis_sz, vis_edf)
    plt.show()
    print(f'Visualizando: {vis_pid} | Crise {vis_sz} | {vis_edf}')
else:
    print('Nenhum espectograma disponivel para visualizacao.')
    print('Execute as celulas de processamento apos os dados estarem disponiveis.')


Visualizando: PN00 | Crise 1 | PN00-1.edf


---
## Pipeline concluído ✅

Os espectogramas estão em `data/processed/spectrograms/<PATIENT_ID>/`  
O manifesto de anotações está em `data/annotations/annotations.csv`

**Próximos passos (conforme marcos do README):**
- `01_exploratory_analysis.ipynb` — EDA e análise da base
- `02_dataset_split.ipynb` — Split treino/val/teste por paciente
- `03_baseline_model.ipynb` — Treinamento do modelo baseline
